# Week 15 — Teaching an Agent to Escape a Frozen Lake

**Theme:** Reinforcement learning basics

Every previous project learned from a fixed dataset of examples. **Reinforcement
learning (RL)** is different: there's no dataset at all — an **agent** takes
**actions** in an **environment**, gets a **reward**, and gradually learns
which actions lead to good outcomes purely through trial and error.

**The task:** `FrozenLake` is a 4x4 grid. `S` = start, `G` = goal (reward +1),
`H` = hole (episode ends, reward 0), `F` = frozen (safe to walk on). The ice is
slippery, so moves don't always go the intended direction — the agent has to
learn a policy that's robust to that randomness.

In [ ]:
!pip install -q gymnasium

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym

np.random.seed(0)

In [ ]:
env = gym.make("FrozenLake-v1", is_slippery=True)
n_states = env.observation_space.n
n_actions = env.action_space.n
print(f"States: {n_states}, Actions: {n_actions} (0=left, 1=down, 2=right, 3=up)")

desc = env.unwrapped.desc.astype(str)
print("The lake:")
for row in desc:
    print(" ", " ".join(row))

## 1. The Q-table

We keep a table `Q[state, action]` estimating "how good is taking this action
from this state, in the long run?" It starts at all zeros — the agent knows
nothing yet.

In [ ]:
Q = np.zeros((n_states, n_actions))
Q.shape

## 2. Q-learning

After each step, we nudge our estimate toward "the reward we just got, plus
the best we think we can do from the new state":

```
Q[s, a] <- Q[s, a] + alpha * (reward + gamma * max(Q[s']) - Q[s, a])
```

- `alpha` (learning rate): how big a step to take toward the new estimate
- `gamma` (discount factor): how much we value future reward vs. immediate reward

We also need to balance **exploration** (try random actions to discover
things) vs. **exploitation** (use what we've already learned) — an
**epsilon-greedy** policy picks a random action with probability `epsilon`,
and the current best action otherwise. We start with high epsilon (explore a
lot) and decay it over training (exploit more as we learn).

In [ ]:
alpha = 0.1        # learning rate -- small and stable beats large and noisy here
gamma = 0.99        # value future reward almost as much as immediate reward
epsilon = 1.0
epsilon_min = 0.01
n_episodes = 30000
# choose the decay so epsilon glides from 1.0 down to epsilon_min by the very last episode
epsilon_decay = (epsilon_min / epsilon) ** (1 / n_episodes)

rewards_per_episode = []

for episode in range(n_episodes):
    state, _ = env.reset()
    done = False
    total_reward = 0

    while not done:
        if np.random.rand() < epsilon:
            action = env.action_space.sample()          # explore
        else:
            action = np.argmax(Q[state])                 # exploit

        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        # Q-learning update
        best_next = np.max(Q[next_state])
        Q[state, action] += alpha * (reward + gamma * best_next - Q[state, action])

        state = next_state
        total_reward += reward

    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    rewards_per_episode.append(total_reward)

print("Training done.")
print(f"Success rate, first 1000 episodes:  {np.mean(rewards_per_episode[:1000]):.1%}")
print(f"Success rate, last 1000 episodes:   {np.mean(rewards_per_episode[-1000:]):.1%}")

## 3. Evaluate the learned policy on its own

The training reward curve still includes a little leftover exploration
(`epsilon_min = 0.01`). To measure what the agent has actually learned, we
run fresh episodes with **no** exploration at all — always take the
Q-table's best action — and average the outcome.

In [ ]:
def evaluate_policy(Q, n_test_episodes=2000):
    successes = 0
    for _ in range(n_test_episodes):
        state, _ = env.reset()
        done = False
        steps = 0
        while not done and steps < 100:
            action = np.argmax(Q[state])   # always exploit -- no randomness
            state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            steps += 1
        successes += reward
    return successes / n_test_episodes

success_rate = evaluate_policy(Q)
print(f"Greedy policy success rate over 2000 test episodes: {success_rate:.1%}")
print("(A policy that acted completely randomly succeeds well under 5% of the time on this map.)")

## 4. Did it actually learn? Plot the reward curve

A single episode's reward is noisy (0 or 1), so we plot a rolling average
success rate across training.

In [ ]:
window = 200
rolling_success = np.convolve(rewards_per_episode, np.ones(window) / window, mode="valid")

plt.figure(figsize=(7, 4))
plt.plot(rolling_success)
plt.title(f"Success Rate Over Training (rolling average, window={window})")
plt.xlabel("Episode")
plt.ylabel("Success rate")
plt.ylim(0, 1)
plt.grid(alpha=0.3)
plt.show()

## 5. Visualize the learned policy

For every non-hole, non-goal square, draw an arrow for the action the agent
now believes is best (`argmax` over that state's row in the Q-table).

In [ ]:
action_arrows = {0: "\u2190", 1: "\u2193", 2: "\u2192", 3: "\u2191"}  # left, down, right, up

fig, ax = plt.subplots(figsize=(5, 5))
grid_size = desc.shape[0]
ax.set_xlim(0, grid_size)
ax.set_ylim(0, grid_size)
ax.set_xticks([])
ax.set_yticks([])

colors = {"S": "lightblue", "F": "white", "H": "black", "G": "gold"}
for r in range(grid_size):
    for c in range(grid_size):
        cell = desc[r, c]
        ax.add_patch(plt.Rectangle((c, grid_size - 1 - r), 1, 1,
                                    facecolor=colors[cell], edgecolor="gray"))
        state_idx = r * grid_size + c
        if cell in ("F", "S"):
            best_action = np.argmax(Q[state_idx])
            ax.text(c + 0.5, grid_size - 1 - r + 0.5, action_arrows[best_action],
                    ha="center", va="center", fontsize=20)
        elif cell == "G":
            ax.text(c + 0.5, grid_size - 1 - r + 0.5, "GOAL", ha="center", va="center", fontsize=9)

ax.set_title("Learned Policy (arrow = best action from each square)")
plt.show()

## Try it yourself

1. **Turn off slipperiness.** Recreate the environment with
   `gym.make("FrozenLake-v1", is_slippery=False)` and retrain — how much
   faster does the success rate reach ~100%, and does the learned policy look
   more like the "obvious" shortest path?
2. **Explore more or less.** Try `epsilon_decay = 0.999` (explores longer) vs.
   `0.99` (exploits sooner) — how does the reward curve's shape change?
3. **Change the discount factor.** Try `gamma = 0.5` (agent barely cares about
   the future) — does it still learn a working policy?
4. **A bigger lake.** Try `gym.make("FrozenLake-v1", map_name="8x8", is_slippery=True)`
   — does the same `n_episodes` still learn a good policy, or does it need
   much more training?